# 08 · PyTorch training with a JAX quantum kernel

`fq.QuantumModule` remains an ordinary PyTorch module. Selecting JAX changes the quantum kernel backend, not the optimizer loop or circuit builder.

In [ ]:
import torch
import flagquantum as fq

def circuit_builder(parameters):
    circuit = fq.Circuit(n_qubits=2)
    circuit.rx(qubit=0, theta=parameters[0])
    circuit.ry(qubit=1, theta=parameters[1])
    return circuit.cx(control=0, target=1)

model = fq.QuantumModule(
    circuit_builder,
    n_parameters=2,
    init=torch.tensor([0.1, -0.2]),
    policy=fq.RuntimePolicy(
        backend='jax',
        observable='z_sum',
        observable_wires=(0, 1),
        allow_backend_fallback=False,
    ),
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
target = torch.tensor(-0.2)
for step in range(3):
    optimizer.zero_grad()
    prediction = model()
    loss = (prediction - target).square().mean()
    loss.backward()
    optimizer.step()
    print({'step': step + 1, 'loss': float(loss.detach())})

print(model.execute().runtime)